# Diabetes Prediction - Numerical Representation

**Portfolio version.** Notebook outputs were cleared to avoid publishing row-level data. The dataset is intentionally excluded from the repository; place `frankfurt_diabetes.csv` in `03_Data/` before running.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import shap
import matplotlib.pyplot as plt

# 1. Membaca dataset
df = pd.read_csv("../03_Data/frankfurt_diabetes.csv")

# 2. Menghilangkan Kolom Glucose
df = df.drop(columns=['Glucose'])

In [ ]:
# 3. Pembersihan Data: Mengganti anomali angka 0 dengan median
features_with_zeros = ['BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for feature in features_with_zeros:
    # Cari median dari data yang bukan 0
    median_val = df[df[feature] != 0][feature].median()
    # Gantikan angka 0 dengan median tersebut
    df[feature] = df[feature].replace(0, median_val)

# Cek duplikasi baris pada dataset mentah
print("Jumlah baris total :", len(df))
print("Jumlah baris unik   :", len(df.drop_duplicates()))
print("Jumlah baris duplikat:", df.duplicated().sum())

# Kalau ada duplikat, lihat contohnya
if df.duplicated().sum() > 0:
    print(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(20))

In [ ]:
# 4. Memisahkan Prediktor (Fitur) dan Target
X = df.drop(columns=['Outcome'])
y = df['Outcome']

In [ ]:
# 5. Train-Test Split (70:30) sesuai standar jurnal
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=6)
X_background = shap.utils.sample(X_train, nsamples=100, random_state=3454) # Background untuk model tertentu

In [ ]:
# Cek apakah ada baris X_test yang identik dengan salah satu baris X_train
merged = X_test.merge(X_train.drop_duplicates(), how="inner")
print("Baris X_test yang identik dengan X_train:", len(merged))

In [ ]:
# 6. Pelatihan Model Random Forest
rf_mod = RandomForestClassifier(
    n_estimators=100,
    random_state=42342
)
rf_mod.fit(X_train, y_train)

In [ ]:
# 7. Evaluasi Model
y_pred = rf_mod.predict(X_test)
print("--- Laporan Klasifikasi Model (Tanpa Glucose) ---")
print(classification_report(y_test, y_pred))

In [ ]:
# 7. Inisialisasi SHAP Explainer (dengan check_additivity=False)
explainer = shap.TreeExplainer(rf_mod, X_train)
rf_log_odds_shap_vals = explainer(X_test, check_additivity=False)

In [ ]:
# VISUALISASI SEMUA PLOT SHAP SESUAI JURNAL
# A. Bar Plot (Global)
shap.plots.bar(rf_log_odds_shap_vals[:,:,1])
plt.show()

In [ ]:
# B. Beeswarm Plot (Global: Arah Hubungan)
shap.plots.beeswarm(rf_log_odds_shap_vals[:,:,1])
plt.show()

In [ ]:
# C. Scatter Plot (Pengaruh satu fitur secara spesifik: Age)
# Kita warnai dengan BMI untuk melihat apakah ada interaksi antara usia dan berat badan
shap.plots.scatter(rf_log_odds_shap_vals[:, "Age", 1], color=rf_log_odds_shap_vals[:, "BMI", 1])
plt.show()

In [ ]:
# D. Waterfall Plot (Penjelasan Individual / Lokal)
# Membedah secara persis alasan model memprediksi pasien pertama (index 0 di data test)
sample_id = 0
shap.plots.waterfall(rf_log_odds_shap_vals[sample_id, :, 1])
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import shap

In [ ]:
# MODEL 2: XGBOOST CLASSIFIER
print("--- Training XGBoost ---")
xgb_mod = XGBClassifier(random_state=342)
xgb_mod.fit(X_train, y_train)

# Evaluasi Model XGBoost
y_pred_xgb = xgb_mod.predict(X_test)
print("--- Laporan Klasifikasi Model XGBoost (Numerik) ---")
print(classification_report(y_test, y_pred_xgb))

# SHAP untuk XGBoost (TreeExplainer)
explainer_xgb = shap.TreeExplainer(xgb_mod)
xgb_shap_vals = explainer_xgb(X_test, check_additivity=False)

# 1. Bar Plot XGBoost
shap.plots.bar(xgb_shap_vals)
plt.show()

# 2. Beeswarm Plot XGBoost
shap.plots.beeswarm(xgb_shap_vals)
plt.show()

# 3. Scatter Plot XGBoost (Fitur Age diwarnai BMI)
shap.plots.scatter(xgb_shap_vals[:, "Age"], color=xgb_shap_vals[:, "BMI"])
plt.show()

# 4. Waterfall Plot XGBoost (Pasien index 0)
shap.plots.waterfall(xgb_shap_vals[0])
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardisasi fitur numerik khusus untuk MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- Training Neural Network (MLP) dengan Standardisasi ---")
nn_mod_scaled = MLPClassifier(hidden_layer_sizes=(10, 10), max_iter=3000, random_state=2, activation="relu")
nn_mod_scaled.fit(X_train_scaled, y_train)

y_pred_nn_scaled = nn_mod_scaled.predict(X_test_scaled)
print("--- Laporan Klasifikasi Model Neural Network (Numerik, Distandardisasi) ---")
print(classification_report(y_test, y_pred_nn_scaled))

In [ ]:
# MODEL 3: NEURAL NETWORK (MLP Classifier)
print("--- Training Neural Network (MLP) ---")
nn_mod = MLPClassifier(hidden_layer_sizes=(10, 10), max_iter=3000, random_state=2, activation="relu")
nn_mod.fit(X_train, y_train)

# Evaluasi Model Neural Network (MLP)
y_pred_nn = nn_mod.predict(X_test)
print("--- Laporan Klasifikasi Model Neural Network (Numerik) ---")
print(classification_report(y_test, y_pred_nn))

# SHAP untuk Neural Network menggunakan Explainer berbasis fungsi probabilitas
def nn_model_proba(x):
    return nn_mod.predict_proba(x)[:, 1]

explainer_nn = shap.Explainer(nn_model_proba, X_background, seed=3454)
nn_shap_vals = explainer_nn(X_test)

# 1. Bar Plot Neural Network
shap.plots.bar(nn_shap_vals)
plt.show()

# 2. Beeswarm Plot Neural Network
shap.plots.beeswarm(nn_shap_vals)
plt.show()

# 3. Scatter Plot Neural Network (Fitur Age diwarnai BMI)
shap.plots.scatter(nn_shap_vals[:, "Age"], color=nn_shap_vals[:, "BMI"])
plt.show()

# 4. Waterfall Plot Neural Network (Pasien index 0)
shap.plots.waterfall(nn_shap_vals[0])
plt.show()